# MLflow Example: Tracking Two Model Training Runs

Corrected version of the course notebook — DagsHub repository: `Candicemamou/mlflow-with-daghub`.

**First use:** execute the installation cell, restart the kernel, then execute the remaining cells in order (starting with the version check). Keep using the existing `Python (MLflow TP)` kernel. No credentials are stored in this notebook.

In [1]:
%pip install "mlflow==2.22.0" "scikit-learn==1.6.1" "numpy==1.26.4" dagshub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
import mlflow
import sklearn
import numpy as np

print(sys.executable)
print("MLflow:", mlflow.__version__)
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
assert mlflow.__version__ == "2.22.0", "Restart the kernel after installation."
assert sklearn.__version__ == "1.6.1", "Restart the kernel after installation."
assert np.__version__ == "1.26.4", "Restart the kernel after installation."

c:\Users\Candice\Documents\S9\Machine Learning in Production\Cours 3\.venv\Scripts\python.exe
MLflow: 2.22.0
scikit-learn: 1.6.1
NumPy: 1.26.4


## 1. Model training and experimentation (without MLflow)

In [3]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [4]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [5]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [6]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


## 2. Now adding MLflow
Connect to the remote MLflow tracking server first. Authorize DagsHub in the browser if prompted.

In [7]:
import mlflow.sklearn
import dagshub
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

dagshub.init(repo_owner="Candicemamou", repo_name="mlflow-with-daghub", mlflow=True)
print("Tracking URI:", mlflow.get_tracking_uri())
assert mlflow.get_tracking_uri().rstrip("/") == "https://dagshub.com/Candicemamou/mlflow-with-daghub.mlflow", "Unexpected tracking URI"

Accessing as Candicemamou

Initialized MLflow to track repo "Candicemamou/mlflow-with-daghub"

Repository Candicemamou/mlflow-with-daghub initialized!

Tracking URI: https://dagshub.com/Candicemamou/mlflow-with-daghub.mlflow


In [8]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [9]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/dc926c5e18874051b65272168559a2e9', creation_time=1795008323730, experiment_id='0', last_update_time=1795008323730, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}>

In [10]:
# Define a function to train a model and log parameters, metrics and the model
ARTIFACT_PATH = "model"

def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        signature = infer_signature(X_train, model.predict(X_train))
        info = mlflow.sklearn.log_model(
            model, artifact_path=ARTIFACT_PATH,
            signature=signature, input_example=X_test[:1]
        )
        print(f"Logged {run_name}: accuracy={accuracy:.4f}")
        print("Model URI:", info.model_uri)
        return run.info.run_id, accuracy, info.model_uri

In [11]:
print("Training and Logging Model 1...")
run_id_1, accuracy_1, model_uri_1 = train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")

Training and Logging Model 1...


Logged random_forest_model_run_1: accuracy=0.9389
Model URI: runs:/9ee789ed7fd640239fd493c291c669a6/model
🏃 View run random_forest_model_run_1 at: https://dagshub.com/Candicemamou/mlflow-with-daghub.mlflow/#/experiments/0/runs/9ee789ed7fd640239fd493c291c669a6
🧪 View experiment at: https://dagshub.com/Candicemamou/mlflow-with-daghub.mlflow/#/experiments/0


In [12]:
print("Training and Logging Model 2...")
run_id_2, accuracy_2, model_uri_2 = train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1.0.2")

Training and Logging Model 2...


Logged random_forest_model_run_V1.0.2: accuracy=0.9722
Model URI: runs:/59ee566bcc5242a8a946caced296af1f/model
🏃 View run random_forest_model_run_V1.0.2 at: https://dagshub.com/Candicemamou/mlflow-with-daghub.mlflow/#/experiments/0/runs/59ee566bcc5242a8a946caced296af1f
🧪 View experiment at: https://dagshub.com/Candicemamou/mlflow-with-daghub.mlflow/#/experiments/0


## 3. Compare results
Open [DagsHub Experiments](https://dagshub.com/Candicemamou/mlflow-with-daghub/experiments) to inspect the two runs. Previous failed runs can remain in the experiment.

In [13]:
print(f"Model 1 accuracy: {accuracy_1:.4f}")
print(f"Model 2 accuracy: {accuracy_2:.4f}")
print("Best model:", "Model 2" if accuracy_2 > accuracy_1 else "Model 1" if accuracy_1 > accuracy_2 else "Tie")

Model 1 accuracy: 0.9389
Model 2 accuracy: 0.9722
Best model: Model 2


## 4. Model Registry
Register both trained models under the same name, in training order. The second registration is the latest version, regardless of accuracy. Run this cell once per pair of training runs. Re-running it creates additional versions.

In [14]:
REGISTERED_MODEL_NAME = "DigitsRandomForest"
version_1 = mlflow.register_model(model_uri_1, REGISTERED_MODEL_NAME)
version_2 = mlflow.register_model(model_uri_2, REGISTERED_MODEL_NAME)
print("Registered versions:", version_1.version, version_2.version)

Successfully registered model 'DigitsRandomForest'.
2026/11/18 15:19:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DigitsRandomForest, version 1
Created version '1' of model 'DigitsRandomForest'.
Registered model 'DigitsRandomForest' already exists. Creating a new version of this model...
2026/11/18 15:19:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DigitsRandomForest, version 2


Registered versions: 1 2


Created version '2' of model 'DigitsRandomForest'.


## 5. Load and test the latest registered model
This is the model that the future Docker build will retrieve via MLflow.

In [15]:
client = MlflowClient()
versions = client.search_model_versions(f"name = '{REGISTERED_MODEL_NAME}'")
ready_versions = [v for v in versions if v.status == "READY"]
assert ready_versions, "No ready model version found"
latest = max(ready_versions, key=lambda v: int(v.version))
latest_uri = f"models:/{REGISTERED_MODEL_NAME}/{latest.version}"
loaded_model = mlflow.sklearn.load_model(latest_uri)
print("Latest model:", latest_uri)
print("Prediction:", loaded_model.predict(X_test[:1]).tolist())
print("Actual digit:", y_test[:1].tolist())

Latest model: models:/DigitsRandomForest/2
Prediction: [6]
Actual digit: [6]


## 6. Serve the trained model (course PDF, step 5)
Use the command printed below in a separate PowerShell terminal with the same virtual environment activated. The tracking server requires authentication; the terminal needs its own MLflow credentials (DagsHub username and access token as environment variables `MLFLOW_TRACKING_USERNAME` and `MLFLOW_TRACKING_PASSWORD`). Never put the token into the notebook or Git. The `--env-manager local` flag uses the existing environment.

In [16]:
print("$env:MLFLOW_TRACKING_URI = '" + mlflow.get_tracking_uri() + "'")
print(f'mlflow models serve -m "runs:/{run_id_2}/{ARTIFACT_PATH}" -p 5003 --env-manager local')

$env:MLFLOW_TRACKING_URI = 'https://dagshub.com/Candicemamou/mlflow-with-daghub.mlflow'
mlflow models serve -m "runs:/59ee566bcc5242a8a946caced296af1f/model" -p 5003 --env-manager local


In [17]:
import mlflow
import mlflow.sklearn
import numpy as np

model = mlflow.sklearn.load_model("models:/DigitsRandomForest/2")

prediction = model.predict(X_test[:5])

print("Predictions:", prediction)
print("Actual values:", y_test[:5])

Predictions: [6 9 3 7 2]
Actual values: [6 9 3 7 2]
